# Practicum: Conditional Memory Recall with Masked Stochastic Attention

In **L15a** and **L15b** you built a **Spiking Hebbian Memory Network (H-Mem)** in which one-shot associations are stored in a synaptic matrix `W_assoc` by Hebbian plasticity, and a key spike train is recalled by the matrix–vector product $\mathbf{v} = \mathbf{W}^{\mathrm{assoc}}\mathbf{k}$. Earlier in the term you saw a sister model: the **modern Hopfield network** of [Ramsauer et al. (2020)](https://arxiv.org/abs/2008.02217), in which stored memories sit on the columns of a matrix `X` and recall is one closed-form softmax-attention update $\mathbf{s} \leftarrow \mathbf{X}\cdot\mathrm{softmax}(\beta\cdot\mathbf{X}^{\top}\mathbf{s})$. **These are the same memory mechanism viewed two ways**: Hebbian when the inverse temperature $\beta \to \infty$, modern-Hopfield at finite $\beta$.

Recent work has pushed this idea one step further. If the modern-Hopfield retrieval rule is the *score function* of a Boltzmann distribution on the network's energy, then injecting Langevin noise into the update turns it into a *training-free generative sampler*. This is **Stochastic Attention (SA)**, used recently to generate small protein families ([Varner, 2026, arXiv:2603.14717](https://arxiv.org/abs/2603.14717)) and synthetic patient cohorts ([Varner et al., 2026, arXiv:2604.07557](https://arxiv.org/abs/2604.07557)) without any model training.

* __Hypothesis__: Suppose we store a balanced subset of MNIST digits as memory patterns and run Stochastic Attention from a random initial state. We should get back recognizable digits, drawn from a smoothed version of the dataset, not literally any image we stored. Now suppose we add a **mask** to the softmax (the same trick GPT uses for causal generation) that zeros out attention to every memory whose label is not the class we want. We should get *only* digits of the chosen class, with no model retraining and no change to the architecture.

This practicum asks you to verify both halves of that hypothesis: that SA over MNIST works as a training-free generator, and that a single boolean mask on the attention softmax is enough to make it class-conditional.

> __Learning Objectives__
>
> By the end of this practicum, you should be able to:
> * __Bridge Hebbian recall and modern-Hopfield retrieval:__ Recognize that the H-Mem read step from L15 and the closed-form softmax-attention update of a modern Hopfield network describe the same associative-memory mechanism. Use the inverse temperature parameter to traverse the soft-to-hard transition between the two views.
> * __Use Stochastic Attention as a training-free generator:__ Add Langevin-style noise to the modern-Hopfield update and treat the result as a sampler from the network's energy landscape. Tune the step size, the noise scale, and the inverse temperature to balance fidelity against novelty in the generated samples.
> * __Steer generation by masking the attention softmax:__ Apply a boolean mask or a finite logit bias to the attention weights to restrict generation to a chosen subset of stored memories. Compare the two conditioning strategies and identify the calibration gap between attention-space conditioning and decoded output.

Let's get started!

___

## Background: From H-Mem to modern Hopfield to Stochastic Attention
The same associative-memory mechanism shows up in three guises across this course.

In **L15a/L15b** you saw the **spiking H-Mem** of Limbacher and Legenstein ([2020](https://proceedings.neurips.cc/paper/2020/file/f6876a9f998f6472cc26708e27444456-Paper.pdf), [2022/2023](https://arxiv.org/abs/2205.11276)). Memories are written into a synaptic matrix `W_assoc` by Hebbian plasticity (the spike-trace outer-product rule), and a key spike train is recalled as $\mathbf{v} = \mathbf{W}^{\mathrm{assoc}}\mathbf{k}$.

The **modern Hopfield network** of [Ramsauer et al. (2020)](https://arxiv.org/abs/2008.02217) replaces `W_assoc` with the explicit memory matrix `X` (one stored pattern per column) and replaces the matrix–vector recall by one closed-form softmax-attention update,
$$\mathbf{s} \;\leftarrow\; \mathbf{X}\cdot\mathrm{softmax}(\beta\cdot\mathbf{X}^{\top}\mathbf{s}).$$
In the limit $\beta \to \infty$ the softmax becomes a one-hot, and the update is exactly the Hebbian read step you used in L15.

**Stochastic Attention (SA)** of [Varner (2026)](https://arxiv.org/abs/2603.14717) treats the modern-Hopfield log-sum-exp energy as a Boltzmann distribution and samples from it by Langevin dynamics,
$$\mathbf{s}_{t+1} \;=\; (1-\eta)\,\mathbf{s}_{t} + \eta\cdot\mathbf{X}\cdot\mathrm{softmax}(\beta\cdot\mathbf{X}^{\top}\mathbf{s}_{t}) \;+\; \sigma\cdot\boldsymbol{\xi}_{t},\qquad \boldsymbol{\xi}_{t}\sim\mathcal{N}(0, \mathbf{I}).$$
With $\sigma = 0$ and $\eta = 1$ this *is* the modern-Hopfield update; with $\sigma > 0$ it draws novel samples from a smoothed family, training-free.

### Why a mask?
Stochastic Attention treats every stored memory equally. To restrict generation to a *subset* of memories (say, all the 7s in MNIST), we add a boolean mask to the softmax: set the logit of every excluded pattern to $-\infty$. This is exactly the masking idea from causal language modeling in transformers, applied to a Hopfield memory rather than a sequence. [Varner (2026, arXiv:2603.20115)](https://arxiv.org/abs/2603.20115) showed that a softer version, a finite scalar bias on the logits of the in-subset patterns, also works, and characterized the **calibration gap** between what the sampler steers toward in attention space and what the decoded output actually looks like.

In this practicum you will implement, run, and explore both versions on MNIST.


## Task 1: Setup, data, and the H-Mem ↔ modern Hopfield bridge
In this task we set up the computational environment, load a balanced subset of MNIST as memory patterns, and verify numerically that the modern-Hopfield retrieval rule reduces to the Hebbian read step from L15a/L15b in the high-$\beta$ limit.

* The `Include.jl` file loads packages and the `src/` modules and seeds the RNG for reproducibility.

In [ ]:
include("Include.jl"); # load deps and src/ modules

### Constants
Before we load data, set a few constants:

* `images_per_class::Int`, how many MNIST images per class to load. The repo ships 30 per class (300 total). Larger pools give richer SA generations but slow down the Langevin loop; 30 per class is plenty for this practicum.
* `β::Float64`, inverse temperature for both the modern-Hopfield recall and SA sampling. Higher $\beta$ makes the softmax sharper (one stored pattern dominates); lower $\beta$ smears attention across multiple patterns.

In [ ]:
images_per_class = 30; # TODO: how many MNIST images per class to memorize (Int between 1 and 30)
β = 8.0;               # TODO: inverse temperature for the softmax (Float64 > 0). Try 1.0, 8.0, 50.0.

### Data
The `data/mnist/<class>/*.jpg` directory tree ships with this repository; there is nothing to download. The helper [`load_mnist_subset(...)`](src/Files.jl) reads up to `images_per_class` images from each of the ten classes, flattens each `28 × 28` image into a length-`784` vector, and stacks them as columns of a single memory matrix `X`. The companion vector `y` records the class label of each column.

The code below stores the memory matrix `X::Matrix{Float64}` (shape `784 × N`) and the class-label vector `y::Vector{Int}` (length `N`) for use in subsequent cells.


In [ ]:
X, y = load_mnist_subset(images_per_class);

__Check__: confirm we have a `784 × N` memory matrix with one label per column and all ten digits present.

In [ ]:
let
    @assert size(X, 1) == 784                                 # 28 × 28 pixels per image
    @assert size(X, 2) == 10 * images_per_class               # 10 classes × images_per_class
    @assert length(y) == size(X, 2)                           # one label per column
    @assert sort(unique(y)) == collect(0:9)                   # all ten digits present
end

__Preview__: peek at one image of each class to confirm we loaded MNIST and not, say, transposed garbage.

In [ ]:
let
    one_per_class = hcat([X[:, findfirst(==(c), y)] for c in 0:9]...)
    image_grid(one_per_class, 10; title = "One image per class")
end

### Build a modern Hopfield network
Construct a `MyModernHopfieldNetworkModel` instance with the MNIST images on its columns and inverse temperature $\beta$.

* We'll use the [`build(...)` factory](src/Factory.jl) over the type [`MyModernHopfieldNetworkModel`](src/Types.jl), which simply stores `(memories = X, β = β)`.

The code below stores the network instance in `mymodel::MyModernHopfieldNetworkModel` for use in subsequent cells.


In [ ]:
mymodel = build(MyModernHopfieldNetworkModel, (
    memories = X, # MNIST images on columns
    β        = β, # inverse temperature
));

### Recall an uncorrupted memory
We seed the recall with one of the stored MNIST images and ask the network to converge by iterating

$$\mathbf{p} = \mathrm{softmax}(\beta \cdot \mathbf{X}^{\top} \mathbf{s}),\qquad \mathbf{s}^{\prime} = \mathbf{X}\,\mathbf{p}$$

until $\lVert \mathbf{p}^{\prime} - \mathbf{p}\rVert_{2}^{2} \le \varepsilon$ or we hit `maxiterations`. With a sharp softmax (high $\beta$), we expect the network to lock onto whichever stored pattern is closest to the seed (often the seed itself).

* `memoryindextorecover::Int`, which column of `X` to use as the seed.

The two cells below store the seed `sₒ::Vector{Float64}` and the integer index `memoryindextorecover::Int`, and then call the recall routine which returns the converged state `ŝₒ::Vector{Float64}`, the per-iteration state dictionary `fₒ::Dict{Int,Vector{Float64}}`, and the per-iteration attention dictionary `pₒ::Dict{Int,Vector{Float64}}`.

In [ ]:
memoryindextorecover = 100; # TODO: pick a column index in 1:size(X, 2)
sₒ = mymodel.X[:, memoryindextorecover]; # the seed
let
    println("Seeding recall with column $(memoryindextorecover); true label = $(y[memoryindextorecover])")
end

In [ ]:
(ŝₒ, fₒ, pₒ) = modern_hopfield_recall(mymodel, sₒ; maxiterations = 10_000, ϵ = 1e-12);
println("Converged in $(length(fₒ)) iterations.")

__Which memory dominated?__ Look at the converged attention vector and find the column that wins the softmax.

In [ ]:
let
    p_final = pₒ[length(fₒ) - 1]
    j_dominant = argmax(p_final)
    println("Dominant stored pattern: column $(j_dominant) (label $(y[j_dominant])) with weight $(round(maximum(p_final), digits=4))")
end

__Visualize__: plot the seed, the recalled state, and the dominant stored memory side by side.

In [ ]:
let
    p_final = pₒ[length(fₒ) - 1]
    j_dominant = argmax(p_final)
    grid = hcat(sₒ, ŝₒ, X[:, j_dominant])
    image_grid(grid, 3; title = "seed | recalled | dominant memory")
end

### The H-Mem ↔ modern Hopfield bridge
Now let's verify the equivalence numerically. As $\beta \to \infty$ the softmax becomes one-hot, the modern-Hopfield update collapses to "find the nearest stored memory", exactly the matrix–vector recall step from L15. We sweep $\beta$ from very cold to very hot and watch the maximum attention weight of the converged recall climb from `1/N` toward `1`.

The code below stores the swept temperatures in `β_grid::Vector{Float64}` and the converged maximum attention weight at each temperature in `recall_max_p::Vector{Float64}`, then plots the relationship.


In [ ]:
β_grid = [0.5, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0, 64.0]
recall_max_p = let
    seed = X[:, memoryindextorecover]
    out = Float64[]
    for βtry in β_grid
        m = build(MyModernHopfieldNetworkModel, (memories = X, β = βtry))
        _, frames, probs = modern_hopfield_recall(m, seed; maxiterations = 10_000, ϵ = 1e-12)
        push!(out, maximum(probs[length(frames) - 1]))
    end
    out
end
plot(β_grid, recall_max_p;
    xscale = :log10, marker = :circle, lw = 2, legend = false,
    xlabel = "β (inverse temperature)",
    ylabel = "max attention weight after convergence",
    title  = "Soft → hard transition: modern Hopfield ⟶ Hebbian read step")

### Discussion
1. As $\beta$ grows, the maximum attention weight of the converged recall climbs toward `1`. Explain in one or two sentences why this is *exactly* the H-Mem read step $\mathbf{v} = \mathbf{W}^{\mathrm{assoc}}\mathbf{k}$ of L15a in disguise. Hint: think about what $\mathrm{softmax}(\beta\cdot\mathbf{X}^{\top}\mathbf{s})$ becomes when $\beta \to \infty$, and what column of $\mathbf{X}\cdot\mathrm{softmax}(\cdot)$ survives in that limit.

Record your answer below and set `did_I_answer_DQ1::Bool` to `true`.


In [ ]:
# -- Put your DQ1 answer here as a comment, or in a sibling markdown cell -- #
did_I_answer_DQ1 = false; # TODO: update the flag value {true | false}

## Task 2: Stochastic Attention on MNIST
With the modern-Hopfield retrieval rule in hand, the next step is small: add Langevin noise to the update and treat what comes out as a sample from the network's Boltzmann distribution. This is the **Stochastic Attention** sampler of [Varner (2026, arXiv:2603.14717)](https://arxiv.org/abs/2603.14717); training-free, no GPU, cost linear in the number of stored patterns.

We use a Langevin-style discrete update of the form

$$\mathbf{s}_{t+1} \;=\; (1-\eta)\,\mathbf{s}_{t} + \eta\cdot\mathbf{X}\cdot\mathrm{softmax}(\beta\cdot\mathbf{X}^{\top}\mathbf{s}_{t}) + \sigma\cdot\boldsymbol{\xi}_{t},\qquad \boldsymbol{\xi}_{t}\sim\mathcal{N}(0, \mathbf{I})$$

where the step size $\eta$ and the noise scale $\sigma$ are treated as independent knobs. The strict Langevin SDE for the modern-Hopfield Boltzmann distribution corresponds to $\sigma = \sqrt{2\eta / \beta}$, but decoupling them lets you probe the fidelity / novelty trade-off directly.

Three knobs you'll tune:

* `step_size`, which plays the role of $\eta \in (0, 1]$, controls how aggressively each step moves toward the Hopfield update; $\eta = 1$ recovers one Hopfield step per iteration.
* `noise_scale`, which plays the role of $\sigma \ge 0$, sets the Gaussian noise standard deviation per step; $\sigma = 0$ reduces to deterministic recall.
* `n_steps`, number of Langevin steps per chain.



__Build the SA sampler__. We reuse the same $\beta$ from Task 1 so the Boltzmann distribution is identical to the one we just visualized; the only thing changing is that we are now *sampling* from it instead of recalling a stored memory.

The code below stores the sampler in `sa_model::MyStochasticAttentionModel` for use in subsequent cells.


In [ ]:
sa_model = build(MyStochasticAttentionModel, (
    memories     = X,
    labels       = y,
    β            = β,    # reuse Task 1's β
    step_size    = 1.0,  # TODO: try 0.25, 0.5, 1.0
    noise_scale  = 0.10, # TODO: try 0.02 (regurgitation), 0.10, 0.30
));

__Generate samples__. Draw 20 fresh samples and look at them. They should *look* like digits drawn from the family, but they are not literally any image we stored.

The code below stores the sample count in `n_samples_unmasked::Int` and the generated samples in `unmasked_samples::Matrix{Float64}` (shape `784 × n_samples_unmasked`).


In [ ]:
n_samples_unmasked = 20;
unmasked_samples = stochastic_attention_sample(sa_model, n_samples_unmasked; n_steps = 200);

In [ ]:
image_grid(unmasked_samples, 10; title = "$(n_samples_unmasked) SA samples (unconditional)")

__Novelty check__. To confirm we are not just memorizing the dataset, compute the Euclidean distance from each generated sample to its nearest stored pattern, and compare to the typical pairwise distance among stored patterns. If the novelty is far below the pairwise spread, we are in regurgitation mode; the temperature is too cold or the step size too small.

In [ ]:
let
    nearest_dist = Float64[]
    for k in 1:size(unmasked_samples, 2)
        s = unmasked_samples[:, k]
        push!(nearest_dist, minimum(norm(s .- X[:, j]) for j in 1:size(X, 2)))
    end
    pairwise_sample = [norm(X[:, i] .- X[:, j]) for i in 1:50 for j in (i + 1):50]
    println("median nearest-stored distance for SA samples: $(round(median(nearest_dist), digits=3))")
    println("median pairwise distance among stored patterns: $(round(median(pairwise_sample), digits=3))")
end

### Discussion
2. Run Task 2 a few times with different combinations of $\beta$, `step_size`, and `noise_scale`. In particular: what happens to the generated samples as $\beta$ grows large (say `64`)? What happens as `noise_scale` grows large (say `0.5`)? Sketch the trade-off; there should be a sweet spot where samples *look* like digits but are not copies of the training set. Explain why this trade-off exists in terms of the SA energy landscape (think about the depth and width of the energy basins around stored patterns at different $\beta$).

Record your answer below and set `did_I_answer_DQ2::Bool` to `true`.


In [ ]:
# -- Put your DQ2 answer here as a comment, or in a sibling markdown cell -- #
did_I_answer_DQ2 = false; # TODO: update the flag value {true | false}

## Task 3: Class-conditional generation by masking the softmax
The whole architecture so far treats every stored memory equally. To draw "give me a 7", we add one boolean mask to the softmax: every stored pattern whose label is *not* `7` gets its attention logit set to $-\infty$, which is exactly zero after the exponential. This is the same masking idea GPT uses for causal generation, restricted here to a Hopfield memory rather than a sequence of tokens.

* `target_class::Int` in `0:9`, which digit to generate.
* `class_mask::Vector{Bool}`, boolean attention mask over stored patterns built as `[yj == target_class for yj in y]`.
* `n_samples_class::Int`, how many samples to draw per class.

We will also compare the hard mask to the **soft logit-bias** version from [Varner (2026, arXiv:2603.20115)](https://arxiv.org/abs/2603.20115): instead of $-\infty$ outside the class, we add a finite positive bias inside the class, leaving every stored memory technically still in play. With a huge bias the soft version approaches the hard version; at zero bias it reduces to the unconditional sampler. The interesting question is what happens *in between*.

The two code cells below set the three globals named above and then call the masked sampler, storing the generated batch in `class_samples::Matrix{Float64}` (shape `784 × n_samples_class`).


In [ ]:
target_class = 7;          # TODO: which digit to generate (0..9)
n_samples_class = 20;      # TODO: how many samples per class
class_mask = [yj == target_class for yj in y];
println("class_mask leaves ", count(class_mask), " of ", length(class_mask), " stored patterns visible")

In [ ]:
class_samples = stochastic_attention_sample(sa_model, n_samples_class;
    n_steps = 200, hard_mask = class_mask);

In [ ]:
image_grid(class_samples, 10;
    title = "Hard-masked SA samples (target_class = $(target_class))")

### Fidelity check
We want a deterministic, training-free way to ask "did this generated sample look like the target class?" The cheapest classifier is the **nearest-mean classifier**: compute the per-class centroid of the stored patterns and label a sample by its closest centroid. No training needed.

We loop over all ten classes, draw `n_samples_class` samples each under the hard mask, and report the fraction that the nearest-mean classifier maps back to the requested class. A perfect hard mask should give very high fractions across the board.

The code below stores the per-class centroid table in `class_means::Dict{Int,Vector{Float64}}` and the per-class hit-rate table in `hardmask_table::DataFrame` for use in subsequent cells.


In [ ]:
class_means = build_class_means(X, y);
hardmask_table = let
    df = DataFrame(target = Int[], hits = Int[], total = Int[], hit_rate = Float64[])
    for c in 0:9
        m = [yj == c for yj in y]
        samples = stochastic_attention_sample(sa_model, n_samples_class;
            n_steps = 200, hard_mask = m)
        labels = [classify_by_nearest_mean(samples[:, k], class_means) for k in 1:n_samples_class]
        h = count(==(c), labels)
        push!(df, (target = c, hits = h, total = n_samples_class, hit_rate = h / n_samples_class))
    end
    pretty_table(df)
    df
end;

### Hard mask vs. soft bias: the calibration gap
A hard mask zeros out off-class logits exactly. A soft bias instead adds a finite scalar bonus to in-class logits and leaves off-class logits in play. With a large bias the soft version approaches the hard version; at zero bias it reduces to the unconditional sampler. In between, the soft version interpolates between the two **in attention space**; but, as [Varner (2026, arXiv:2603.20115)](https://arxiv.org/abs/2603.20115) shows, that interpolation does not always land where you would naively expect in pixel space. The discrepancy is the **calibration gap**.

We sweep the bias and see what happens.

The code below stores the bias-versus-hit-rate table in `softbias_table::DataFrame` for use in subsequent cells.


In [ ]:
softbias_table = let
    df = DataFrame(bias = Float64[], target = Int[], hits = Int[], total = Int[], hit_rate = Float64[])
    in_class = [yj == target_class for yj in y]
    for b in [0.0, 1.0, 5.0, 20.0, 100.0]
        sb_vec = Float64.([m ? b : 0.0 for m in in_class])
        samples = stochastic_attention_sample(sa_model, n_samples_class;
            n_steps = 200, soft_bias = sb_vec)
        labels = [classify_by_nearest_mean(samples[:, k], class_means) for k in 1:n_samples_class]
        h = count(==(target_class), labels)
        push!(df, (bias = b, target = target_class, hits = h, total = n_samples_class, hit_rate = h / n_samples_class))
    end
    pretty_table(df)
    df
end;

In [ ]:
# Visualize one batch from each bias setting side-by-side at a fixed target class.
let
    biases = [0.0, 1.0, 5.0, 20.0, 100.0]
    n_per = 5
    grid_cols = Vector{Vector{Float64}}()
    in_class = [yj == target_class for yj in y]
    for b in biases
        sb_vec = Float64.([m ? b : 0.0 for m in in_class])
        s = stochastic_attention_sample(sa_model, n_per; n_steps = 200, soft_bias = sb_vec)
        for k in 1:n_per
            push!(grid_cols, s[:, k])
        end
    end
    G = reduce(hcat, grid_cols)
    image_grid(G, n_per; title = "Soft-bias sweep on target_class = $(target_class). Rows: bias = " * join(string.(biases), ", "))
end

### Discussion
3. Compare the hard-mask hit-rate table with the soft-bias hit-rate table.
   * For a bias just above zero, the hit rate barely moves; for very large bias, it climbs toward the hard-mask value. Why is this? Think about the magnitude of the inner-product term $\beta\cdot\mathbf{X}^{\top}\mathbf{s}$ versus the additive bias.
   * Even at bias = 100, the soft-bias version may not reach 100% hit rate, while the hard mask does. What is being lost? Connect this to the **calibration gap** of [Varner (2026, arXiv:2603.20115)](https://arxiv.org/abs/2603.20115): the conditioning is exact at the level of the sampler's softmax weights, but the *decoded* sample can disagree because the inner-product term steers the chain toward whichever attractor is closest to the current state, regardless of class.

Record your answer below and set `did_I_answer_DQ3::Bool` to `true`.


In [ ]:
# -- Put your DQ3 answer here as a comment, or in a sibling markdown cell -- #
did_I_answer_DQ3 = false; # TODO: update the flag value {true | false}

## Fun (totally optional) directions we could go with this idea
This was a proof-of-concept exploration of masked Stochastic Attention on MNIST. Real applications are well beyond image-based digits, and the three papers below (all of which were born from this class) sketch where the idea goes next.

### Maybe a different application?
* __Small protein families__. A small alignment of <100 sequences from a single Pfam family is exactly the regime where deep generative models overfit, but where Stochastic Attention has linear-in-alignment-size cost on a laptop. See [Varner, *Training-Free Generation of Protein Sequences from Small Family Alignments via Stochastic Attention*, arXiv:2603.14717](https://arxiv.org/abs/2603.14717).
* __Conditioning on a binding screen__. If you have a small functional subset of a family (say, hits from a binding screen) and want to bias generation toward proteins that share that property, the multiplicity-weighted soft bias of this practicum's Task 3 is exactly the recipe; see [Varner, *Conditioning Protein Generation via Hopfield Pattern Multiplicity*, arXiv:2603.20115](https://arxiv.org/abs/2603.20115). The same paper diagnoses the calibration-gap phenomenon you saw in DQ3.
* __Synthetic patient cohorts__. Replacing "stored protein sequence" with "stored longitudinal patient record" lets SA augment small clinical cohorts of <30 patients in domains like pregnancy and rare disease without retraining. See [Varner, Bravo, McBride, Orfeo, Bernstein, *Validated Synthetic Patient Generation for Small Longitudinal Cohorts*, arXiv:2604.07557](https://arxiv.org/abs/2604.07557).

### Other things to try
* __Causal mask__: instead of masking by class, set up a causal mask (`mask[j] = (j ≤ t)` at time step `t`) so the chain only attends to "earlier" memories, a Hopfield analogue of GPT's autoregressive generation.
* __Different inverse temperatures inside vs. outside the mask__: replace the scalar $\beta$ with a vector of per-pattern temperatures. This decouples *how confident* the sampler is about each memory from *which* memories are visible.
* __Compare to the spiking H-Mem of L15b__. The hard-mask SA produces a class-conditional generator from a static memory; the spiking H-Mem produces one-shot recall from a dynamically-written memory. Sketch how you would turn one into the other.

## Key Takeaways
* **The Hebbian read step is the cold limit of softmax attention:** When the inverse temperature grows large, the softmax over stored patterns collapses to a one-hot indicator of the nearest memory. The modern-Hopfield retrieval rule and the H-Mem matrix-vector recall therefore describe the same operation at different temperatures.
* **Stochastic Attention generates novel samples without training:** Injecting Gaussian noise into the modern-Hopfield update produces a Langevin-style sampler whose stationary distribution is the Hopfield Boltzmann measure over stored memories. Cost is linear in the number of stored patterns, and the sampler runs on a laptop with no gradient descent and no tuned weights.
* **Hard masks condition cleanly while soft biases do not:** A boolean mask over the attention softmax delivers near-perfect class-conditional generation by exactly excluding off-class memories. A finite scalar bias on in-class logits steers attention but rarely overcomes the inner-product term that dominates the softmax, leaving a calibration gap between attention-space conditioning and pixel-space output.


## Summary
This practicum connected the Spiking Hebbian Memory Network of L15 to the modern Hopfield network of Ramsauer et al. (2020) and to the Stochastic Attention sampler of Varner (2026), then introduced a single boolean mask on the attention softmax as a route to class-conditional generation on MNIST. The same memory mechanism therefore powers retrieval, generation, and conditional generation, and the choice of mask versus bias is a practical, training-free knob with measurable trade-offs.


## Tests
In the code block below, we check some values in your notebook and give you feedback on which items are correct or different. Unhide the code block (if you are curious) to see how the tests are set up.

In [ ]:
let
    @testset verbose = true "CHEME 5820 Practicum S2026" begin

        @testset "Task 1: Setup, Data, and the H-Mem ↔ modern Hopfield bridge" begin
            @test _DID_INCLUDE_FILE_GET_CALLED == true
            @test isnothing(images_per_class) == false
            @test isnothing(β) == false
            @test β > 0
            @test size(X, 1) == 784
            @test size(X, 2) == 10 * images_per_class
            @test length(y) == size(X, 2)
            @test sort(unique(y)) == collect(0:9)
            @test isnothing(mymodel) == false
            @test mymodel.β == β
            @test isnothing(memoryindextorecover) == false
            @test 1 <= memoryindextorecover <= size(X, 2)
            @test length(fₒ) > 0
            @test length(pₒ) > 0
            @test length(recall_max_p) == length(β_grid)
            @test all(0.0 .<= recall_max_p .<= 1.0)
            @test recall_max_p[end] >= recall_max_p[1]   # max attention weight is non-decreasing in β
            @test did_I_answer_DQ1 == true
        end

        @testset "Task 2: Stochastic Attention on MNIST" begin
            @test isnothing(sa_model) == false
            @test sa_model.β == β
            @test sa_model.step_size > 0
            @test sa_model.noise_scale >= 0
            @test size(unmasked_samples) == (784, n_samples_unmasked)
            @test all(isfinite, unmasked_samples)
            @test did_I_answer_DQ2 == true
        end

        @testset "Task 3: Masked SA = class-conditional generation" begin
            @test 0 <= target_class <= 9
            @test length(class_mask) == size(X, 2)
            @test count(class_mask) == count(==(target_class), y)
            @test size(class_samples) == (784, n_samples_class)
            @test isnothing(class_means) == false
            @test length(class_means) == 10
            # Hard mask should land in the requested class >70% of the time, averaged across classes.
            @test mean(hardmask_table.hit_rate) > 0.7
            # Calibration gap: at this β, even the largest soft-bias hit rate stays well
            # below the hard-mask average; that gap is the central finding of DQ3 and
            # of Varner (2026, arXiv:2603.20115). Tightening β or shifting embeddings
            # can close the gap; that's a stretch direction.
            @test mean(hardmask_table.hit_rate) > maximum(softbias_table.hit_rate)
            @test did_I_answer_DQ3 == true
        end
    end
end;